In [20]:
from gradescope_auth import create_new_user, login_with_token, login_temporary, save_profile_for_token
from concurrent.futures import ThreadPoolExecutor
%load_ext autoreload 
%autoreload 2 

token = 'jzGzeA1Z7e_izljQXT4Mki4NsxUTKOhrqD30L-iwtTs'

if token:
    with ThreadPoolExecutor(max_workers=1) as executor:
        conn = executor.submit(lambda: login_with_token(token)).result()
else:
    with ThreadPoolExecutor(max_workers=1) as executor:
        conn, temp_profile_dir = executor.submit(login_temporary).result()
    token = create_new_user()
    save_profile_for_token(temp_profile_dir, token)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [125]:
courses = conn.account.get_courses()
# course_id = [k for (k,v) in courses['instructor'].items() if v.name=='COSI 21a'][0]
course_id = [k for (k,v) in courses['instructor'].items() if v.name=='TEST'][0]
assignment_id = conn.account.get_assignments(course_id)[-1].assignment_id

course_id, assignment_id

('1311586', '8237331')

In [ ]:
from utils import * 
students, max_student_name_length = get_student_info(conn, course_id)
instructors = get_instructor_info(conn, course_id)
student_mapping = get_user_mapping(students)
instructor_mapping = get_user_mapping(instructors)
users = students + instructors
user_mapping = student_mapping | instructor_mapping
questions, questions_order = get_assignment_questions(conn, course_id, assignment_id)
raw_submissions_metadata = get_raw_submissions_metadata(conn, course_id, assignment_id)
grades_metadata = get_grades_metadata(conn, course_id, assignment_id, instructors, users)
student_to_assignment_submissions = get_student_to_assignment_submissions(users, raw_submissions_metadata, grades_metadata)
grader_by_question_submission = get_grader_by_question_submission(conn, course_id, questions)
question_to_submissions = get_question_to_question_submissions(conn, course_id, questions)
comments, total_scores, student_to_question_to_question_submission = get_raw_data_by_question_submission(conn, course_id, users, questions, question_to_submissions, student_to_assignment_submissions)
grade_breakdowns = get_grade_breakdowns(users, questions, comments, total_scores, student_to_question_to_question_submission, grader_by_question_submission, questions_order)
users_with_grades = [u for u in users if grades_metadata[u.email_address]['submitted']]
